# Mamba SOH training — Kaggle GPU

Notebook train 2 model từ cùng 1 checkout code:
- **Part A — Standard model v1.4** (window=30, GH-54: +cycle_count +soc_percent, 6 features) — dùng cho production inference
- **Part B — Long-sequence model v2.2** (L=4096, GH-25 feature ablation) — dùng cho long-range degradation experiment

**Trước khi chạy:**
1. Settings → Accelerator → **GPU P100/T4**
2. + Add Data → dataset NASA chứa `cleaned_dataset/metadata.csv` + `cleaned_dataset/data/*.csv`
3. (repo private) Add-ons → Secrets → tạo `GITHUB_TOKEN` = GitHub PAT
4. **Push code lên GitHub trước** (branch chứa các thay đổi bạn muốn train — vd `dev` sau khi merge GH-54) — Kaggle clone từ remote, không thấy local uncommitted changes

**v2.0 improvements của long-seq (so với v1.x MAE ceiling 2.24%):**
| Thay đổi | Chi tiết |
|---|---|
| +2 features (6 total) | IC curve (dQ/dV, SOH indicator) + discharge progress (phase channel) — base 4: bỏ current_load/voltage_load (GH-25 ablation) |
| `PatchDegradationEncoder` | Local RMS/P2P/std/kurtosis per 16-step patch → per-token local context |
| 2-layer FiLM (SiLU) | Deeper feature conditioning vs single Linear |
| `CosineAnnealingWarmRestarts` | Periodic LR restarts escape sharp minima (replaces ReduceLROnPlateau) |
| `SmoothL1Loss(beta=0.02)` | Clamp gradient for >2% residuals; reduce label noise impact |
| Discharge-weighted attention | Last channel (discharge_progress) steers attn toward end-of-discharge |
| `LONG_SEQ_STRIDE=64` | ~4400 train windows (2× v1.x) |
| Scheduler reset at final stage | v1.x bug: warmup transitions reduced LR before final stage started |

> ⚠️ `mode="reduce-overhead"` (CUDA Graphs) **không dùng được khi training** — dùng `mode="default"`.
> `scaler_long.pkl` (6 features) được tạo bởi `preprocess_long.py` — độc lập với `scaler.pkl` (4 features, standard model).

## 1 — GPU check

In [ ]:
!nvidia-smi
import torch
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: bật GPU ở Settings -> Accelerator -> GPU P100/T4')

## 2 — Clone branch (push code lên GitHub trước khi chạy cell này)

In [ ]:
import subprocess
BRANCH  = 'dev'   # đổi thành branch của bạn nếu chưa merge (vd feat/GH-54-...) — PHẢI push lên GitHub trước khi chạy cell này
REPO    = '/kaggle/working/ai-module'
URL_PUB = 'https://github.com/GSU26SE55/ai-module.git'
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('GITHUB_TOKEN')
    url = f'https://{token}@github.com/GSU26SE55/ai-module.git'
except Exception as e:
    print('No GITHUB_TOKEN secret -> thử public clone:', e)
    url = URL_PUB
subprocess.run(['rm', '-rf', REPO])
subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', url, REPO], check=True)
subprocess.run(['git', '-C', REPO, 'remote', 'set-url', 'origin', URL_PUB])  # xoá token khỏi remote
print('Branch:', subprocess.check_output(['git','-C',REPO,'branch','--show-current']).decode().strip())
print('Commit:', subprocess.check_output(['git','-C',REPO,'log','-1','--oneline']).decode().strip())

## 3 — Dependencies (torch đã có sẵn trên Kaggle)

In [ ]:
%pip install -q scipy scikit-learn joblib pandas
import scipy, sklearn; print('scipy', scipy.__version__, '| sklearn', sklearn.__version__)

In [ ]:
## 3b — (Optional) Install official mamba-ssm CUDA backend
# Nếu thành công: train nhanh hơn ~3-5x, ít VRAM hơn ở L=4096.
# Nếu fail (version mismatch): tự fallback về pure-PyTorch — không cần làm gì thêm.
import subprocess, sys, torch

USE_OFFICIAL_MAMBA = True   # đổi False nếu muốn dùng pure-PyTorch

if USE_OFFICIAL_MAMBA and torch.cuda.is_available():
    print("Installing causal-conv1d + mamba-ssm (build ~5-10 phút lần đầu)...")
    r1 = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "causal-conv1d>=1.1.0"], capture_output=True)
    r2 = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mamba-ssm"],            capture_output=True)
    try:
        import mamba_ssm
        print(f"mamba-ssm {mamba_ssm.__version__} OK — official CUDA Mamba sẽ được dùng (3-5x nhanh hơn)")
    except ImportError:
        print("mamba-ssm install failed — sẽ fallback về pure-PyTorch MambaBlock tự động")
        USE_OFFICIAL_MAMBA = False
else:
    print("Skipping mamba-ssm install (USE_OFFICIAL_MAMBA=False hoặc không có GPU)")
    USE_OFFICIAL_MAMBA = False

print(f"USE_OFFICIAL_MAMBA = {USE_OFFICIAL_MAMBA}")

## 4 — Tìm NASA dataset + vào repo

In [ ]:
import os, subprocess
REPO = '/kaggle/working/ai-module'
found = [f for f in subprocess.check_output(['find','/kaggle/input','-name','metadata.csv']).decode().splitlines() if f]
assert found, 'Khong thay metadata.csv — + Add Data dataset NASA cleaned_dataset'
DATASET = os.path.dirname(found[0])
os.chdir(REPO)
print('DATASET    :', DATASET)
print('has data/  :', os.path.isdir(f'{DATASET}/data'))
print('cwd        :', os.getcwd())
print('scaler.pkl :', os.path.isfile('models/weights/scaler.pkl'), '(committed, preprocess_long reuse)')

## 5 — Part A: Standard model v1.4 (window=30, GH-54) — preprocess

Tạo `data/processed/{train,val,test}.pt` (6 features: 4 base + cycle_count + soc_percent) + `models/weights/scaler.pkl` (4-feature) + `feature_scaler.pkl`.

In [ ]:
import os; os.chdir('/kaggle/working/ai-module')
!python scripts/preprocess.py --data-dir "{DATASET}" --output-dir data/processed

## 6 — Part A: Full training (v1.4)

Model nhỏ (D_MODEL=64, 2 layer) — train được cả trên CPU, nhưng GPU nhanh hơn nhiều. `--epochs 100` với early-stop `patience=15` (xem `scripts/train.py`).

In [ ]:
import os; os.chdir('/kaggle/working/ai-module')
!python scripts/train.py --data-dir data/processed --epochs 100 --log-dir logs/training

## 7 — Part A: Kết quả + đóng gói artifact (v1.4)

In [ ]:
import os, glob, shutil, torch, sys
os.chdir('/kaggle/working/ai-module')
sys.path.insert(0, '/kaggle/working/ai-module')
from src.core.config import MAMBA_PATH, ISO_FOREST_PATH, SCALER_PATH, FEATURE_SCALER_PATH

logs = sorted(glob.glob('logs/training/train_*.log'), key=os.path.getmtime)
if logs:
    print('Log:', logs[-1]); print('-'*50)
    !grep -E "Test MAE|Test RMSE|Saved Mamba|Saved IsolationForest" "{logs[-1]}"
if os.path.isfile(MAMBA_PATH):
    c = torch.load(MAMBA_PATH, map_location='cpu', weights_only=False)
    print('-'*50)
    print(f"version={c['version']} window={c['window_size']} input_features={c['input_features']}")
    print(f"Test MAE={c['test_mae']:.4f}%  RMSE={c['test_rmse']:.4f}%")
else:
    print(f'Checkpoint not found: {MAMBA_PATH}')

os.makedirs('/kaggle/working/out_std', exist_ok=True)
for f in [MAMBA_PATH, ISO_FOREST_PATH, SCALER_PATH, FEATURE_SCALER_PATH]:
    if os.path.isfile(f): shutil.copy2(f, '/kaggle/working/out_std/'); print('copied', f)
shutil.make_archive('/kaggle/working/mamba_v1.4_artifacts', 'zip', '/kaggle/working/out_std')
print('\nDownload: Output tab -> mamba_v1.4_artifacts.zip')
print('Commit 4 artifacts (soh_mamba_v1.4.pth, isolation_forest_v1.4.pkl, scaler.pkl, feature_scaler.pkl) vao dev + note MAE/RMSE.')

## 8 — Part B: Long-sequence (L=4096) v2.2 — preprocess (ghép cycle → chuỗi 4096)

Tạo `data/processed_long/{train,val,test}.pt` + `models/weights/feature_scaler_long.pkl`.

In [ ]:
import os; os.chdir('/kaggle/working/ai-module')
!python scripts/preprocess_long.py --data-dir "{DATASET}" --output-dir data/processed_long

## 9 — Part B: Smoke test (1 epoch/stage) — kiểm tra pipeline trước

In [ ]:
import os; os.chdir('/kaggle/working/ai-module')
# Smoke test: 1 epoch/stage — verify pipeline trước khi chạy full
# --patch-size 16 --patch-stride 16: test patch pipeline (same flags as full training)
!python scripts/train.py --long --stage-epochs 1 --final-epochs 1 --micro-batch 8 --benchmark \
    --patch-size 16 --patch-stride 16

## 10 — Part B: Full training (v2.2 — feature ablation 4 base, cùng flags run B)

Run trước đạt **MAE 2.06% / RMSE 2.46%** nhưng các đòn bẩy generalization còn **TẮT**. Run này bật hết:

| Flag | Run cũ (2.06%) | Run này (B) | Vì sao |
|---|---|---|---|
| `--jitter` | 0.0 (off) | **0.0075** | Input noise = Tikhonov regularization (Bishop 1995) → robust cell-to-cell |
| `--dropout` | 0.2 | **0.3** | Giảm overfit pin train |
| `--weight-decay` | 1e-5 | **3e-4** | AdamW decoupled decay (Loshchilov 2019) |
| `--cosine-t0` | 10 | **80 (= final-epochs)** | 1 chu kỳ anneal mượt; hết early-stop ở epoch 30 |
| `--swa` | — | **bật** | Average weight đuôi → optimum phẳng, giảm variance trên test 1-pin (Izmailov 2018) |
| `--benchmark` | ON | **BỎ** | Deterministic, reproduce được (rule seed-42) |
| `--weighted-loss` | ON | ON (giữ) | Upweight vùng SOH≤80% = vùng B0048 |

**Chẩn đoán run 2.06%:** early-stop ở epoch 30 (best ~epoch 15) → final stage chỉ train ~15 epoch ở L=4096, regularization off. Run B sửa cả hai → kỳ vọng cross < 2.0%. SWA là lớp bảo hiểm (chỉ giữ nếu thắng trên val).

> SWA cần final stage chạy đủ dài mới có gì để average → **bắt buộc** `--cosine-t0 80 --final-epochs 80` (không early-stop).

In [ ]:
import os; os.chdir('/kaggle/working/ai-module')
# v2.2 — feature ablation (base 6->4, bỏ current_load/voltage_load) + giữ nguyên flags run B:
#   Regularization (đang OFF ở run cũ):  --jitter 0.0075 --dropout 0.3 --weight-decay 3e-4
#   Schedule fix:  --cosine-t0 80 == --final-epochs 80  -> 1 chu kỳ cosine anneal mượt tới cuối,
#                  KHÔNG còn early-stop sớm ở epoch 30 (final_patience = max(15, 85) = 85)
#   SWA:           --swa  -> trung bình weight 25% epoch cuối (optimum rộng hơn, giảm variance
#                  trên test 1-pin B0048). Chỉ giữ nếu thắng best-checkpoint trên VAL (không leak test).
#   Deterministic: BỎ --benchmark  -> cuDNN deterministic, reproduce được (rule seed-42)
#   Giữ: --weighted-loss (vùng EOL = nơi B0048 nằm) + --patch-stride 8 (510 tokens)
official_flag = '--official-mamba' if USE_OFFICIAL_MAMBA else ''
!python scripts/train.py --long {official_flag} --compile --num-workers 4 \
    --patch-size 16 --patch-stride 8 \
    --weighted-loss --eol-weight-scale 2.0 \
    --stage-epochs 3 \
    --cosine-t0 80 --final-epochs 80 \
    --weight-decay 3e-4 --dropout 0.3 --jitter 0.0075 \
    --swa --swa-start-frac 0.75 \
    --micro-batch 8 --accum-steps 4 --eval-batch 16

## 11 — Part B: Kết quả + đóng gói artifact (v2.2)

In [ ]:
import os, glob, shutil, torch, sys
os.chdir('/kaggle/working/ai-module')
sys.path.insert(0, '/kaggle/working/ai-module')
from src.core.config import LONG_MAMBA_PATH, LONG_SCALER_PATH, LONG_FEATURE_SCALER_PATH

logs = sorted(glob.glob('logs/training/train_*.log'), key=os.path.getmtime)
if logs:
    print('Log:', logs[-1]); print('-'*50)
    !grep -E "Test MAE|Test RMSE|\[stage|Saved long|Patch:|Final stage|Optim:|SWA|Final weights|benchmark" "{logs[-1]}"
if os.path.isfile(LONG_MAMBA_PATH):
    c = torch.load(LONG_MAMBA_PATH, map_location='cpu', weights_only=False)
    print('-'*50)
    ps, ss = c.get('patch_size', 1), c.get('patch_stride', 1)
    num_tokens = (c['seq_len'] - ps) // ss + 1 if ps > 1 else c['seq_len']
    print(f"seq_len={c['seq_len']} patch={ps}s{ss} → {num_tokens} tokens | pooling={c['pooling']}")
    print(f"weights={c.get('final_weights','?')} | swa={c.get('swa')} | dropout={c.get('dropout')} | wd={c.get('weight_decay')} | jitter={c.get('jitter')}")
    print(f"Test MAE={c['test_mae']:.4f}%  RMSE={c['test_rmse']:.4f}%")
else:
    print(f"Checkpoint not found: {LONG_MAMBA_PATH}")
# package artifacts to download
os.makedirs('/kaggle/working/out', exist_ok=True)
for f in [LONG_MAMBA_PATH, LONG_SCALER_PATH, LONG_FEATURE_SCALER_PATH]:
    if os.path.isfile(f): shutil.copy2(f, '/kaggle/working/out/'); print('copied', f)
shutil.make_archive('/kaggle/working/mamba_long_artifacts', 'zip', '/kaggle/working/out')
print('\nDownload: Output tab -> mamba_long_artifacts.zip')
print('Commit 3 artifacts vào branch + điền MAE vào PR.')

## Nếu MAE > 2%

**Các cải thiện đã tích hợp (stride=64, scheduler reset, stage_epochs=5)** nhằm phá vỡ ceiling 2.24%:
- `LONG_SEQ_STRIDE = 64` (giảm từ 128) → ~4400 train windows (2×)
- Final stage reset LR + fresh `ReduceLROnPlateau` → tránh LR bị giảm do spike khi chuyển stage warmup
- `stage_epochs=5` (tăng từ 3) → model học ổn định hơn ở mỗi độ dài trước khi chuyển sang dài hơn

Nếu SAU KHI áp các fix trên MAE vẫn > 2%, thử **hạ L=2048**:
```python
import re, pathlib
p = pathlib.Path('/kaggle/working/ai-module/src/core/config.py')
p.write_text(re.sub(r'LONG_SEQ_LEN\s*=\s*4096', 'LONG_SEQ_LEN    = 2048', p.read_text()))
```

Với L=2048 + `--compile --benchmark`, full training hoàn thành trong **~25-40 phút** (vs 2-3h với L=4096).